In [ ]:
import torch, torchvision
from torchvision.models import resnet50, resnet18,  ResNet50_Weights, ResNet18_Weights
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
import numpy as np

In [31]:
# pre trained models with imagenet
model_teacher = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)


couche_fc = model_teacher.fc
couche_classif = torch.nn.Linear(couche_fc.in_features, 10)
model_teacher.fc = couche_classif
model_teacher = model_teacher.to("cuda")

model_student  = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
couche_fc_student = model_student.fc
couche_classif_student = torch.nn.Linear(couche_fc_student.in_features, 10)
model_student.fc = couche_classif_student
model_student = model_student.to("cuda")

In [40]:
params_teacher = model_teacher.parameters()
total= 0
for i in params_teacher:
    t = i.requires_grad
    if t: 
        total += i.numel()


params_student = model_student.parameters()
total_student = 0
for i in params_student:
    t = i.requires_grad
    if t:
        total_student += i.numel()
print(total_student)

<generator object Module.parameters at 0x7e3ff35fbca0>
11181642


In [35]:
# pretraitment of data 
data_prep = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform= data_prep
)
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform= data_prep
)

train_subset, val_dataset = random_split(
    train_dataset,
    [40000, 10000]
)

train_dataloader = DataLoader(
    train_subset, 
    batch_size=32,
    shuffle=True,
    num_workers=2
)
validation_dataloader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle= False,
    num_workers=2
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model_teacher.parameters(),
      lr=0.001
    )

optimizer_student = torch.optim.Adam(
    model_student.parameters(),
    lr=0.001
)

In [27]:
for epoch in range(10):
    val_loss = 0
    l = 0
    model_teacher.train()
    for batch_X, batch_Y in train_dataloader:
        batch_X = batch_X.to("cuda")
        batch_Y = batch_Y.to("cuda")
        
        optimizer.zero_grad()
        logits = model_teacher(batch_X)
        loss = criterion(logits, batch_Y)
        loss.backward()
        optimizer.step()

    model_teacher.eval()
    with torch.no_grad():
        for batch_X, batch_Y in validation_dataloader:
            batch_X = batch_X.to("cuda")
            batch_Y = batch_Y.to("cuda")
            prediction = model_teacher(batch_X)
            val_loss += criterion(prediction, batch_Y).item()
            true_label = batch_Y
            classes_predites = prediction.argmax(dim=1)
            comparaison = true_label == classes_predites

            l += comparaison.sum().item()

        acc = l / len(val_dataset)

        val_loss = val_loss / len(validation_dataloader)


    print(f"Epoch {epoch + 1} - Validation Loss : {val_loss:.4f}, accuracy: {acc:.2%}")

Epoch 1 - Validation Loss : 0.5084, accuracy: 82.53%
Epoch 2 - Validation Loss : 0.4530, accuracy: 84.79%
Epoch 3 - Validation Loss : 0.5437, accuracy: 82.54%
Epoch 4 - Validation Loss : 0.4216, accuracy: 86.38%
Epoch 5 - Validation Loss : 0.5028, accuracy: 84.74%
Epoch 6 - Validation Loss : 0.4252, accuracy: 86.92%
Epoch 7 - Validation Loss : 0.4847, accuracy: 86.80%
Epoch 8 - Validation Loss : 0.4407, accuracy: 87.46%
Epoch 9 - Validation Loss : 0.5087, accuracy: 86.51%
Epoch 10 - Validation Loss : 0.4701, accuracy: 87.57%


In [28]:
torch.save(model_teacher.state_dict(), "/kaggle/working/teacher_resnet50.pth")

In [29]:
model_teacher.eval()
l_test = 0
test_loss = 0
with torch.no_grad():
    for batch_X, batch_Y in test_dataloader:
        batch_X = batch_X.to("cuda")
        batch_Y = batch_Y.to("cuda")
        prediction = model_teacher(batch_X)
        test_loss += criterion(prediction, batch_Y).item()
        true_label = batch_Y
        classes_predites = prediction.argmax(dim=1)
        comparaison = true_label == classes_predites

        l_test += comparaison.sum().item()

    acc = l_test / len(test_dataset)

    test_loss = test_loss / len(test_dataloader)
print(f"Test Loss : {test_loss:.4f}, accuracy: {acc:.2%}")

Test Loss : 0.4697, accuracy: 87.60%


In [37]:
for epoch in range(10):
    val_loss = 0
    l = 0
    model_student.train()
    for batch_X, batch_Y in train_dataloader:
        batch_X = batch_X.to("cuda")
        batch_Y = batch_Y.to("cuda")
        
        optimizer_student.zero_grad()
        logits = model_student(batch_X)
        loss = criterion(logits, batch_Y)
        loss.backward()
        optimizer_student.step()

    model_student.eval()
    with torch.no_grad():
        for batch_X, batch_Y in validation_dataloader:
            batch_X = batch_X.to("cuda")
            batch_Y = batch_Y.to("cuda")
            prediction = model_student(batch_X)
            val_loss += criterion(prediction, batch_Y).item()
            true_label = batch_Y
            classes_predites = prediction.argmax(dim=1)
            comparaison = true_label == classes_predites

            l += comparaison.sum().item()

        acc = l / len(val_dataset)

        val_loss = val_loss / len(validation_dataloader)


    print(f"Epoch {epoch + 1} - Validation Loss : {val_loss:.4f}, accuracy: {acc:.2%}")

Epoch 1 - Validation Loss : 0.4038, accuracy: 86.19%
Epoch 2 - Validation Loss : 0.3988, accuracy: 86.82%
Epoch 3 - Validation Loss : 0.4252, accuracy: 86.71%
Epoch 4 - Validation Loss : 0.3686, accuracy: 88.43%
Epoch 5 - Validation Loss : 0.4212, accuracy: 87.13%
Epoch 6 - Validation Loss : 0.4446, accuracy: 87.97%
Epoch 7 - Validation Loss : 0.3569, accuracy: 89.90%
Epoch 8 - Validation Loss : 0.4407, accuracy: 88.26%
Epoch 9 - Validation Loss : 0.4871, accuracy: 88.52%
Epoch 10 - Validation Loss : 0.4582, accuracy: 88.76%


In [38]:
torch.save(model_student.state_dict(), "/kaggle/working/student_resnet18.pth")

In [39]:
model_student.eval()
l_test = 0
test_loss = 0
with torch.no_grad():
    for batch_X, batch_Y in test_dataloader:
        batch_X = batch_X.to("cuda")
        batch_Y = batch_Y.to("cuda")
        prediction = model_student(batch_X)
        test_loss += criterion(prediction, batch_Y).item()
        true_label = batch_Y
        classes_predites = prediction.argmax(dim=1)
        comparaison = true_label == classes_predites

        l_test += comparaison.sum().item()

    acc = l_test / len(test_dataset)

    test_loss = test_loss / len(test_dataloader)
print(f"Test Loss : {test_loss:.4f}, accuracy: {acc:.2%}")

Test Loss : 0.4762, accuracy: 88.93%


In [44]:
model_teacher.load_state_dict(
    torch.load(
        "/kaggle/working/teacher_resnet50.pth",
        map_location="cuda",
        weights_only=True
    )
)

model_student = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

model_student.fc = torch.nn.Linear(
    model_student.fc.in_features, 10
)

model_student = model_student.to("cuda")

optimizer_student = torch.optim.Adam(
    model_student.parameters(),
    lr=0.001
)

In [ ]:
model_teacher.eval()
for params in model_teacher.parameters():
    params.requires_grad = False
    
criterion = torch.nn.MSELoss()
for epoch in range(10):
    total_mse = 0
    l = 0
    model_student.train()
    
    for batch_X, batch_Y in train_dataloader:
        batch_X = batch_X.to("cuda")
        batch_Y = batch_Y.to("cuda")
    
        optimizer_student.zero_grad()
        logits = model_student(batch_X)
        with torch.no_grad():
            prediction_teacher = model_teacher(batch_X)
        mse = criterion(logits, prediction_teacher)
        total_mse += mse.item()
        mse.backward()
        optimizer_student.step()

    model_student.eval()
    with torch.no_grad():
        for batch_X, batch_Y in validation_dataloader:
            batch_X = batch_X.to("cuda")
            batch_Y = batch_Y.to("cuda")
            prediction = model_student(batch_X)
            true_label = batch_Y
            classes_predites = prediction.argmax(dim=1)
            comparaison = batch_Y == classes_predites
            l += comparaison.sum().item()
    
        acc = l / len(val_dataset)
    
    total_mse = total_mse / len(train_dataloader)

print(f"Epoch {epoch + 1} - Train MSE : {total_mse:.4f}, "f"Validation Accuracy : {acc:.2%}")